# Translation & speech quality harness: a short walkthrough

This notebook runs the harness on a slice of the demo set and shows how to read one flag back to its evidence.
The real interface is the CLI (`python -m bodhan_eval ...`); this is only a guided tour. Keys come from `.env` (see `.env.example`).

In [ ]:
import json, subprocess, sys
from pathlib import Path

# 1. Score the first 30 demo items (cached model outputs make reruns free)
lines = Path("data/demo.jsonl").read_text(encoding="utf-8").splitlines()[:30]
Path("outputs").mkdir(exist_ok=True)
Path("outputs/slice.jsonl").write_text("
".join(lines) + "
", encoding="utf-8")
subprocess.run([sys.executable, "-m", "bodhan_eval", "--dataset", "outputs/slice.jsonl", "--langs", "hi,ta,bn",
                "--out", "outputs/report_slice", "--consistency-sample", "5"], check=True)

## 2. Headline numbers

In [ ]:
print(Path("outputs/report_slice/summary.md").read_text(encoding="utf-8")[:2500])

## 3. One flag, end to end: category, span, flagged text, and the evidence from every check that agreed

In [ ]:
rows = [json.loads(l) for l in open("outputs/report_slice/items.jsonl", encoding="utf-8")]
flagged = [r for r in rows if r["flags"]]
r = flagged[0]
f = r["flags"][0]
print(r["id"], r["lang"], "|", f["category"], f["severity"], f["confidence"], "| span", f["span"])
print("source:     ", r["source"])
print("flagged:    ", f.get("text"))
print("back-transl:", r["back_translation"])
for e in f["evidence"]:
    print(" -", e["check"], e["confidence"], e["message"])

## 4. Where to go next
- `calibration.json`: precision/recall per category and per check, tuned thresholds, cross-validated metrics, judge consistency
- `review_queue.csv`: the uncertain items; label them and convert with `python -m bodhan_eval.review`
- `dashboard.html`: filter by language, category and severity